# 🖼️ Panoramic Image Stitching
### Using SIFT · Lowe's Ratio Test · RANSAC · Homography · Warp Perspective

---

This notebook walks through the complete pipeline for stitching multiple overlapping images into a single wide-angle panorama — built from scratch using classical computer vision techniques.

**Pipeline Overview:**
```
Input Images  →  SIFT Feature Detection  →  Feature Matching (Lowe's Ratio Test)
             →  Homography (RANSAC)       →  Warp Perspective  →  Panorama Output
```

## Step 1 — Install & Import Libraries

In [ ]:
# Install required packages (run once)
# !pip install opencv-contrib-python numpy imutils Pillow matplotlib

In [ ]:
import cv2
import numpy as np
import imutils
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

print(f"OpenCV version : {cv2.__version__}")
print(f"NumPy  version : {np.__version__}")
print("All libraries loaded successfully ✓")

---
## Step 2 — Helper: Display Images in Notebook

OpenCV uses BGR colour order. Matplotlib expects RGB, so we convert before showing.

In [ ]:
def show(images, titles=None, figsize=(18, 5), cmap=None):
    """Display one or more OpenCV images side-by-side in the notebook."""
    if not isinstance(images, (list, tuple)):
        images = [images]
    if titles is None:
        titles = [f"Image {i+1}" for i in range(len(images))]

    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]

    for ax, img, title in zip(axes, images, titles):
        if len(img.shape) == 2:          # grayscale
            ax.imshow(img, cmap="gray")
        else:                             # BGR → RGB
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

print("show() helper ready ✓")

---
## Step 3 — Load Input Images

We use the **Taj Mahal** sample images included in the project.
Images must be in **left → right** order as they were taken.

In [ ]:
# ── Configure paths here ────────────────────────────────────────────
IMAGE_PATHS = [
    "inputs/tajm1.jpg",
    "inputs/tajm2.jpg",
    "inputs/tajm3.jpg",
    "inputs/tajm4.jpg",
]
# ────────────────────────────────────────────────────────────────────

TARGET_WIDTH  = 400
TARGET_HEIGHT = 400

raw_images = []
for path in IMAGE_PATHS:
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f"Cannot read: {path}")
    img = imutils.resize(img, width=TARGET_WIDTH)
    img = imutils.resize(img, height=TARGET_HEIGHT)
    raw_images.append(img)
    print(f"  Loaded: {path}  →  shape: {img.shape}")

print(f"\n✓ {len(raw_images)} images loaded")
show(raw_images, [os.path.basename(p) for p in IMAGE_PATHS])

---
## Step 4 — SIFT Feature Detection

**SIFT (Scale-Invariant Feature Transform)** finds distinctive points (keypoints) in an image and describes them with a 128-dimensional vector.  
These descriptors are robust to:
- ✅ Scale changes
- ✅ Rotation
- ✅ Illumination differences

> **Why SIFT?**  
> When we take overlapping photos, they may be taken at slightly different distances or angles. SIFT still finds and matches the same real-world points across both images.

> **Requires:** `opencv-contrib-python` (not plain `opencv-python`)

In [ ]:
def detect_features(image):
    """
    Detect SIFT keypoints and compute descriptors.

    Args:
        image: grayscale or BGR image

    Returns:
        keypoints : float32 array of (x, y) positions
        features  : float32 descriptor array  [N x 128]
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
    sift = cv2.SIFT_create()
    kps, features = sift.detectAndCompute(gray, None)
    keypoints = np.float32([kp.pt for kp in kps])
    return keypoints, features, kps   # also return raw kps for visualization


# Detect on the first two images and visualize
kp1_coords, feat1, kp1_raw = detect_features(raw_images[0])
kp2_coords, feat2, kp2_raw = detect_features(raw_images[1])

print(f"Image 1 → {len(kp1_coords)} keypoints detected")
print(f"Image 2 → {len(kp2_coords)} keypoints detected")

# Draw keypoints (circles = location, size = scale, line = orientation)
vis1 = cv2.drawKeypoints(raw_images[0], kp1_raw, None,
                          flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
vis2 = cv2.drawKeypoints(raw_images[1], kp2_raw, None,
                          flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

show([vis1, vis2],
     [f"SIFT Keypoints – Image 1 ({len(kp1_coords)} pts)",
      f"SIFT Keypoints – Image 2 ({len(kp2_coords)} pts)"])

---
## Step 5 — Feature Matching with Lowe's Ratio Test

We use **Brute-Force KNN Matching** (k=2) to find the 2 nearest neighbours for every descriptor in Image A.

**Lowe's Ratio Test** (David Lowe, 2004):  
A match is only kept if:
```
distance_to_best_match  <  lowe_ratio × distance_to_second_best_match
```
This filters out ambiguous matches where two features look equally similar.

In [ ]:
LOWE_RATIO = 0.75   # lower = stricter filtering

def match_features(featA, featB, lowe_ratio=0.75):
    """
    Brute-force KNN matching + Lowe's ratio test.

    Returns:
        valid_matches : list of (trainIdx, queryIdx) tuples
    """
    matcher = cv2.DescriptorMatcher_create("BruteForce")
    all_matches = matcher.knnMatch(featA, featB, 2)

    valid = []
    for pair in all_matches:
        if len(pair) == 2 and pair[0].distance < pair[1].distance * lowe_ratio:
            valid.append((pair[0].trainIdx, pair[0].queryIdx))

    return valid


valid_matches = match_features(feat1, feat2, LOWE_RATIO)

total = sum(len(m) for m in cv2.DescriptorMatcher_create("BruteForce").knnMatch(feat1, feat2, 2))
print(f"Total candidate matches : {total}")
print(f"Valid matches (after Lowe's ratio test) : {len(valid_matches)}")
print(f"Filtered out : {total - len(valid_matches)} ambiguous matches")

In [ ]:
# Visualize the valid matches using cv2.drawMatches
sift_for_vis = cv2.SIFT_create()
kps1_vis = sift_for_vis.detect(cv2.cvtColor(raw_images[0], cv2.COLOR_BGR2GRAY))
kps2_vis = sift_for_vis.detect(cv2.cvtColor(raw_images[1], cv2.COLOR_BGR2GRAY))

dm_matches = [cv2.DMatch(q, t, 0) for (t, q) in valid_matches[:60]]  # show top 60

match_vis = cv2.drawMatches(
    raw_images[0], kps1_vis,
    raw_images[1], kps2_vis,
    dm_matches, None,
    matchColor=(0, 255, 0),
    singlePointColor=(200, 200, 200),
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

show([match_vis], [f"Top {len(dm_matches)} Valid Feature Matches (green lines)"], figsize=(18, 5))

---
## Step 6 — Compute Homography with RANSAC

A **Homography** is a 3×3 matrix that describes a perspective transformation from one plane to another.  
It answers: *"how do I warp Image A so that it aligns perfectly with Image B?"*

**RANSAC (Random Sample Consensus)** makes this robust:  
Instead of using all matched points (some may be wrong), RANSAC:
1. Randomly picks 4 point pairs
2. Computes a homography from them
3. Counts how many other points agree (inliers)
4. Repeats many times, keeping the best homography


In [ ]:
MAX_RANSAC_THRESHOLD = 4.0   # max reprojection error (pixels) to be counted as inlier

def compute_homography(kpA, kpB, valid_matches, max_threshold=4.0):
    """
    Build point arrays from matched keypoints and compute homography via RANSAC.

    Returns:
        (homography_matrix, inlier_status_array)
    """
    if len(valid_matches) <= 4:
        raise ValueError("Not enough matches to compute homography (need > 4).")

    ptsA = np.float32([kpA[i] for (_, i) in valid_matches])
    ptsB = np.float32([kpB[i] for (i, _) in valid_matches])

    H, status = cv2.findHomography(ptsA, ptsB, cv2.RANSAC, max_threshold)
    return H, status


H, status = compute_homography(kp1_coords, kp2_coords, valid_matches, MAX_RANSAC_THRESHOLD)

inliers  = int(status.sum())
outliers = len(status) - inliers

print("Homography Matrix H:")
print(np.round(H, 4))
print(f"\nTotal matched points : {len(status)}")
print(f"RANSAC Inliers       : {inliers}  ✓")
print(f"RANSAC Outliers      : {outliers}  ✗  (filtered out)")

---
## Step 7 — Warp Perspective & Blend

Using the homography matrix H, we **warp Image A** into the coordinate frame of **Image B**.  
Then we paste Image B directly onto the left side of the result — giving us the stitched panorama.


In [ ]:
def warp_and_stitch(imageA, imageB, H):
    """
    Warp imageA using H and paste imageB on the left to create the panorama.

    Returns:
        stitched image (before cropping)
    """
    width  = imageA.shape[1] + imageB.shape[1]
    height = max(imageA.shape[0], imageB.shape[0])

    warped = cv2.warpPerspective(imageA, H, (width, height))
    warped[0:imageB.shape[0], 0:imageB.shape[1]] = imageB   # paste imageB on top
    return warped


stitched_raw = warp_and_stitch(raw_images[0], raw_images[1], H)

show([raw_images[1], raw_images[0], stitched_raw],
     ["Image B (left)", "Image A (right)", "After warpPerspective (with black border)"],
     figsize=(18, 5))

---
## Step 8 — Crop Black Borders

After warping, the areas with no image data appear as black borders.  
We detect all non-black pixels and crop to their bounding rectangle.

In [ ]:
def crop_black_borders(image):
    """
    Remove black borders introduced by warpPerspective.
    Finds the bounding box of all non-zero pixels and crops to it.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)

    col_sums = np.sum(thresh, axis=0)
    row_sums = np.sum(thresh, axis=1)

    non_zero_cols = np.where(col_sums > 0)[0]
    non_zero_rows = np.where(row_sums > 0)[0]

    if len(non_zero_cols) == 0 or len(non_zero_rows) == 0:
        return image

    x0, x1 = non_zero_cols[0], non_zero_cols[-1]
    y0, y1 = non_zero_rows[0], non_zero_rows[-1]

    return image[y0:y1+1, x0:x1+1]


stitched_clean = crop_black_borders(stitched_raw)

print(f"Before crop : {stitched_raw.shape}")
print(f"After crop  : {stitched_clean.shape}")

show([stitched_raw, stitched_clean],
     ["Before Cropping", "After Cropping ✓"],
     figsize=(18, 5))

---
## Step 9 — The Panaroma Class (Full Implementation)

All steps above are packaged into a reusable `Panaroma` class.

In [ ]:
class Panaroma:
    """
    Panoramic Image Stitcher.

    Stitches two images using:
      SIFT  →  Lowe's Ratio Test  →  RANSAC Homography  →  Warp Perspective
    """

    def image_stitch(self, images, lowe_ratio=0.75, max_Threshold=4.0, match_status=False):
        """Main entry point. images = [imageB (left), imageA (right)]."""
        (imageB, imageA) = images

        if imageA is None or imageB is None:
            raise ValueError("One or both input images are None.")

        kpA, featA, _ = self.detect_feature_and_keypoints(imageA)
        kpB, featB, _ = self.detect_feature_and_keypoints(imageB)

        if featA is None or featB is None:
            print("[ERROR] Feature extraction failed.")
            return None

        result = self.match_keypoints(kpA, kpB, featA, featB, lowe_ratio, max_Threshold)
        if result is None:
            print("[ERROR] Not enough matches found.")
            return None

        matches, H, status = result
        panorama = self.get_warp_perspective(imageA, imageB, H)
        panorama[0:imageB.shape[0], 0:imageB.shape[1]] = imageB
        panorama = self.crop_black_borders(panorama)

        if match_status:
            vis = self.draw_matches(imageA, imageB, kpA, kpB, matches, status)
            return panorama, vis

        return panorama

    # ── Feature Detection ──────────────────────────────────────────

    def detect_feature_and_keypoints(self, image):
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
        sift = cv2.SIFT_create()
        kps_raw, features = sift.detectAndCompute(gray, None)
        if len(kps_raw) == 0:
            return np.array([]), None, []
        kps = np.float32([kp.pt for kp in kps_raw])
        return kps, features, kps_raw

    # ── Matching ───────────────────────────────────────────────────

    def get_all_possible_matches(self, featA, featB):
        matcher = cv2.DescriptorMatcher_create("BruteForce")
        return matcher.knnMatch(featA, featB, 2)

    def get_all_valid_matches(self, all_matches, lowe_ratio):
        valid = []
        for pair in all_matches:
            if len(pair) == 2 and pair[0].distance < pair[1].distance * lowe_ratio:
                valid.append((pair[0].trainIdx, pair[0].queryIdx))
        return valid

    def compute_homography(self, ptsA, ptsB, max_Threshold):
        return cv2.findHomography(ptsA, ptsB, cv2.RANSAC, max_Threshold)

    def match_keypoints(self, kpA, kpB, featA, featB, lowe_ratio, max_Threshold):
        all_m  = self.get_all_possible_matches(featA, featB)
        valid  = self.get_all_valid_matches(all_m, lowe_ratio)
        if len(valid) <= 4:
            return None
        ptsA = np.float32([kpA[i] for (_, i) in valid])
        ptsB = np.float32([kpB[i] for (i, _) in valid])
        H, status = self.compute_homography(ptsA, ptsB, max_Threshold)
        if H is None:
            return None
        return valid, H, status

    # ── Warping ────────────────────────────────────────────────────

    def get_warp_perspective(self, imageA, imageB, H):
        w = imageA.shape[1] + imageB.shape[1]
        h = max(imageA.shape[0], imageB.shape[0])
        return cv2.warpPerspective(imageA, H, (w, h))

    # ── Visualization ──────────────────────────────────────────────

    def draw_matches(self, imageA, imageB, kpA, kpB, matches, status):
        hA, wA = imageA.shape[:2]
        hB, wB = imageB.shape[:2]
        canvas = np.zeros((max(hA, hB), wA + wB, 3), dtype="uint8")
        canvas[0:hA, 0:wA] = imageA
        canvas[0:hB, wA:]  = imageB
        for ((tIdx, qIdx), s) in zip(matches, status):
            if s == 1:
                ptA = (int(kpA[qIdx][0]),        int(kpA[qIdx][1]))
                ptB = (int(kpB[tIdx][0]) + wA,   int(kpB[tIdx][1]))
                cv2.line(canvas, ptA, ptB, (0, 255, 0), 1)
        return canvas

    # ── Post-processing ────────────────────────────────────────────

    def crop_black_borders(self, image):
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        _, thresh = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
        col_sums = np.sum(thresh, axis=0)
        row_sums = np.sum(thresh, axis=1)
        nzc = np.where(col_sums > 0)[0]
        nzr = np.where(row_sums > 0)[0]
        if len(nzc) == 0 or len(nzr) == 0:
            return image
        return image[nzr[0]:nzr[-1]+1, nzc[0]:nzc[-1]+1]


print("Panaroma class defined ✓")

---
## Step 10 — Stitch All Images Together

We stitch images **right-to-left**, progressively accumulating the panorama:
```
img3 + img4  →  partial
img2 + partial  →  partial
img1 + partial  →  final panorama
```

In [ ]:
def stitch_all(images, lowe_ratio=0.75, max_threshold=4.0):
    """
    Stitch a list of images (left → right order) into one panorama.

    Returns:
        (panorama_image, matched_points_image)
    """
    panorama_engine = Panaroma()
    n = len(images)

    if n < 2:
        raise ValueError("Need at least 2 images.")

    print(f"Stitching {n} images...")

    print(f"  Step 1: Stitching image {n-1} + image {n}")
    output = panorama_engine.image_stitch(
        [images[n - 2], images[n - 1]],
        lowe_ratio=lowe_ratio,
        max_Threshold=max_threshold,
        match_status=True
    )

    if output is None:
        raise RuntimeError("First stitch failed. Ensure images overlap.")

    result, matched = output

    for i in range(n - 2):
        idx = n - i - 3
        print(f"  Step {i+2}: Adding image {idx+1} to panorama")
        output = panorama_engine.image_stitch(
            [images[idx], result],
            lowe_ratio=lowe_ratio,
            max_Threshold=max_threshold,
            match_status=True
        )
        if output is None:
            raise RuntimeError(f"Failed at image {idx+1}. Check order/overlap.")
        result, matched = output

    print("\n✓ Stitching complete!")
    return result, matched


# Run the full pipeline
panorama_result, matched_points = stitch_all(raw_images)

print(f"Final panorama size: {panorama_result.shape}")

---
## Step 11 — View Results

In [ ]:
# Show keypoint matches visualization
show([matched_points], ["Keypoint Matches (RANSAC Inliers = green lines)"], figsize=(18, 5))

In [ ]:
# Show the final panorama
show([panorama_result], ["✅ Final Stitched Panorama"], figsize=(18, 6))

---
## Step 12 — Save Output

In [ ]:
os.makedirs("output", exist_ok=True)

panorama_path = "output/panorama_image.jpg"
matches_path  = "output/matched_points.jpg"

cv2.imwrite(panorama_path, panorama_result)
cv2.imwrite(matches_path,  matched_points)

print(f"✓ Panorama saved  →  {panorama_path}")
print(f"✓ Matches saved   →  {matches_path}")

---
## Step 13 — Try Your Own Images

Just change the paths in the cell below and re-run from Step 10.

In [ ]:
# ── Change these to your own image paths ──────────────────────────
MY_IMAGES = [
    "inputs/tajm1.jpg",
    "inputs/tajm2.jpg",
    "inputs/tajm3.jpg",
    "inputs/tajm4.jpg",
]
# Other datasets in data/ folder:
#   data/beach/  |  data/nature/  |  data/room/  |  data/Hill/
# ─────────────────────────────────────────────────────────────────

my_imgs = []
for p in MY_IMAGES:
    img = cv2.imread(p)
    if img is None:
        raise FileNotFoundError(p)
    img = imutils.resize(img, width=400)
    img = imutils.resize(img, height=400)
    my_imgs.append(img)

my_panorama, my_matches = stitch_all(my_imgs)

show([my_panorama], ["My Panorama"], figsize=(18, 6))

cv2.imwrite("output/my_panorama.jpg", my_panorama)
print("Saved → output/my_panorama.jpg")

---
## Summary

| Step | Algorithm | Purpose |
|------|-----------|----------|
| 1 | **SIFT** | Detect scale/rotation-invariant keypoints and 128-D descriptors |
| 2 | **Brute-Force KNN** | Find 2 nearest neighbour matches for every descriptor |
| 3 | **Lowe's Ratio Test** | Filter ambiguous matches (keep only clear winners) |
| 4 | **RANSAC** | Robustly compute homography, ignoring outlier matches |
| 5 | **Warp Perspective** | Transform image A into image B's coordinate frame |
| 6 | **Black Border Crop** | Remove empty regions introduced by warping |

---
**References**
- D. Lowe, *Distinctive Image Features from Scale-Invariant Keypoints*, IJCV 2004
- M. Fischler & R. Bolles, *Random Sample Consensus*, Communications of the ACM 1981
- OpenCV Documentation: https://docs.opencv.org